# Retrieval Evaluation

We compare three retrieval strategies against the ground-truth question set generated in `ground-truth-generation.ipynb`, using Hit Rate and MRR:

1. **Keyword search** — TF-IDF via `minsearch.Index` (the current production search in `rag.py`).
2. **Vector search** — cosine similarity over embeddings from a local ONNX MiniLM model (`coffee_assistant/embedder.py`), no API calls.
3. **Hybrid search** — Reciprocal Rank Fusion (RRF) of both.

The winning approach is wired into `rag.py`'s `search()`.

In [1]:
import pandas as pd
df_question = pd.read_csv('../data/ground-truth-retrieval.csv')

In [2]:
df_question.head()

,chunk_id,question
0,604727_0,What is the origin of coffee consumption accor...
1,604727_0,Which two main types of coffee beans are menti...
2,604727_0,How has the global coffee industry economicall...
3,604727_0,What are some common ways coffee can be prepar...
4,604727_0,Where are coffee plants primarily cultivated a...


In [3]:
ground_truth = df_question.to_dict(orient='records')

In [4]:
ground_truth[0]

{'chunk_id': '604727_0',
 'question': 'What is the origin of coffee consumption according to the article?'}

## Metrics: Hit Rate and MRR

`evaluate` runs a search function over every ground-truth question and checks whether the correct `chunk_id` shows up in the results.

In [5]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

In [6]:
from tqdm.auto import tqdm
def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        chunk_id = q['chunk_id']
        results = search_function(q)
        relevance = [d['chunk_id'] == chunk_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

## 1. Keyword search (baseline)

`retrieval.keyword_search()` is the plain TF-IDF search, evaluated in isolation from the hybrid pipeline below — note this is deliberately *not* `rag.search()`, since that now runs hybrid search in production (see the wiring at the bottom of this notebook).

In [7]:
import sys
sys.path.append('..')

from coffee_assistant import rag, retrieval

In [8]:
keyword_results = evaluate(ground_truth, lambda q: retrieval.keyword_search(q['question'], num_results=10))
keyword_results

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.5511904761904762, 'mrr': 0.33508865583270403}

## 2. Vector search

Embeddings are built by `retrieval.get_vector_index()` — every chunk embedded with a local ONNX MiniLM model (`coffee_assistant/embedder.py`, no PyTorch/API calls), rebuilt fresh in memory each run. `retrieval.vector_search()` ranks by cosine similarity (the embeddings are L2-normalized, so a dot product is enough).

In [9]:
vector_results = evaluate(ground_truth, lambda q: retrieval.vector_search(q['question'], 10))
vector_results

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.7404761904761905, 'mrr': 0.4602897455278408}

## 3. Hybrid search (Reciprocal Rank Fusion)

`retrieval.hybrid_search()` merges the keyword and vector result lists by summing `1 / (k + rank)` per method for each `chunk_id` (`retrieval.rrf`), then ranks by the combined score. `k` is a smoothing constant from the RRF literature.

In [10]:
# k=60 here (the RRF literature default) purely to demonstrate the tuning point below —
# retrieval.hybrid_search()'s actual default is k=1, the winner picked further down.
hybrid_results = evaluate(ground_truth, lambda q: retrieval.hybrid_search(q['question'], k=60))
hybrid_results

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.7857142857142857, 'mrr': 0.41587474804736835}

### Tuning the RRF `k` parameter

The common default `k=60` comes from search literature where rankings run into the thousands. Our lists only have 10 candidates each (rank 0–9), so at `k=60` the score `1/(k+rank)` barely differs between rank 0 and rank 9 — RRF ends up mostly rewarding "appears in both lists" and ignoring *where*. A much smaller `k` restores that signal. Sweep a few values to check the effect on Hit Rate and MRR.

In [11]:
for k in [1, 5, 10, 50, 60, 100, 200]:
    result = evaluate(ground_truth,
                      lambda q, k=k: retrieval.hybrid_search(q['question'], k))
    print(f"Hybrid search (k={k}): Hit Rate={result['hit_rate']:.4f}, MRR={result['mrr']:.4f}")

  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=1): Hit Rate=0.7857, MRR=0.4302


  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=5): Hit Rate=0.7857, MRR=0.4221


  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=10): Hit Rate=0.7857, MRR=0.4166


... (4 outputs left out)

  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=100): Hit Rate=0.7857, MRR=0.4159


  0%|          | 0/2520 [00:00<?, ?it/s]

Hybrid search (k=200): Hit Rate=0.7857, MRR=0.4159


## Results

| Method | Hit Rate | MRR |
|---|---|---|
| Keyword | 0.551 | 0.335 |
| Vector | 0.740 | 0.460 |
| Hybrid (k=60, RRF default) | 0.786 | 0.416 |
| **Hybrid (k=1)** | **0.786** | **0.430** |

Hybrid search wins on Hit Rate. Vector search wins on MRR, so it ranks the right chunk higher when both find it. `rag.py` puts all 10 results into the prompt, so finding the right chunk matters more than its position. **Hybrid search with `k=1` stays in production**, wired into `coffee_assistant/retrieval.py` (`hybrid_search`'s default) and called by `rag.py`'s `search()`.

The embedder reads 256 tokens per chunk (`MAX_TOKENS` in `embedder.py`). With the 128 from `tokenizer.json`, it only read the first ~600 of 2000 characters, and vector search reached 0.615 Hit Rate (hybrid 0.732).

In [12]:
# Final pick: hybrid search, k=1 — this is retrieval.hybrid_search()'s actual default,
# and what rag.py's search() calls in production.
evaluate(ground_truth, lambda q: retrieval.hybrid_search(q['question'], k=1))

  0%|          | 0/2520 [00:00<?, ?it/s]

{'hit_rate': 0.7857142857142857, 'mrr': 0.4301900667674488}

## 4. Query rewriting

Before searching, the app asks gpt-4o-mini to fix the question's spelling and wording (`rag.rewrite_query()`). Only the search uses the new version; the answer still uses the user's own question.

Each rewrite is an API call, so we use 500 questions instead of all 2520. These questions have no typos, so here we just check that rewriting doesn't make search worse.

In [13]:
sample_rw = df_question.sample(n=500, random_state=1).to_dict(orient='records')

rewritten = {}
for q in tqdm(sample_rw):
    rewritten[q['question']] = rag.rewrite_query(q['question'])[0]

  0%|          | 0/500 [00:00<?, ?it/s]

In [14]:
clean_no_rewrite = evaluate(sample_rw, lambda q: retrieval.hybrid_search(q['question']))
clean_with_rewrite = evaluate(sample_rw, lambda q: retrieval.hybrid_search(rewritten[q['question']]))

print('clean, hybrid, no rewrite:  ', clean_no_rewrite)
print('clean, hybrid, with rewrite:', clean_with_rewrite)

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

clean, hybrid, no rewrite:   {'hit_rate': 0.808, 'mrr': 0.45619920634920613}
clean, hybrid, with rewrite: {'hit_rate': 0.786, 'mrr': 0.4324730158730157}


### Same questions, with typos

Real users make typos. A user typed "coffeine", and search found nothing about caffeine. To test this, we add typos to the same 500 questions (swapping two letters in some words) and compare search with and without rewriting.

In [15]:
import random

def add_typos(text, rng, rate=0.3):
    # swap two adjacent inner letters in ~30% of words with 5+ characters
    words = text.split()
    for i, w in enumerate(words):
        if len(w) >= 5 and rng.random() < rate:
            j = rng.randrange(1, len(w) - 2)
            words[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    return ' '.join(words)

rng = random.Random(42)
sample_typo = [{**q, 'question': add_typos(q['question'], rng)} for q in sample_rw]

rewritten_typo = {}
for q in tqdm(sample_typo):
    rewritten_typo[q['question']] = rag.rewrite_query(q['question'])[0]

  0%|          | 0/500 [00:00<?, ?it/s]

In [16]:
for q in sample_typo[:5]:
    print('typo:     ', q['question'])
    print('rewritten:', rewritten_typo[q['question']])
    print()

typo:      How does the speed of pouirng wtaer affect the taste of the coffee when usnig a drip mtehod?
rewritten: How does the speed of pouring water affect the flavor of coffee when using the drip brewing method?

typo:      What two tpyes of coffee are combined in the process of slow-brewing with espresso, and what prupose does this serve?
rewritten: What two types of coffee are combined in the process of slow brewing with espresso, and what purpose does this serve?

typo:      How does the caffiene content in Araibca coffee comapre to Robusta coffee?
rewritten: How does the caffeine content in Arabica coffee compare to that in Robusta coffee?

typo:      What is the general strength variation of an americano based on espresso shots?
rewritten: What is the typical variation in strength of an Americano based on the number of espresso shots used?

typo:      What collaborative efforts have prominent cfofee companies in Vietnam undertaken to improve coffee production?
rewritten: What c

In [17]:
typo_no_rewrite = evaluate(sample_typo, lambda q: retrieval.hybrid_search(q['question']))
typo_with_rewrite = evaluate(sample_typo, lambda q: retrieval.hybrid_search(rewritten_typo[q['question']]))

print('typos, hybrid, no rewrite:  ', typo_no_rewrite)
print('typos, hybrid, with rewrite:', typo_with_rewrite)

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

typos, hybrid, no rewrite:   {'hit_rate': 0.718, 'mrr': 0.3862103174603173}
typos, hybrid, with rewrite: {'hit_rate': 0.788, 'mrr': 0.440535714285714}
